1. Read the file using spark dataframe reader API
2. Add metadata columns - Source file, Ingestion timestamp
3. Write to bronze delta table

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/02.bronze-helpers

In [0]:
source_file = f"{landing_folder_path}/{v_batch_id}/constructors.json"
table_name = f"{catalog_name}.{bronze_schema}.constructors"

In [0]:
constructors_schema = """
    constructorId STRING,
    name STRING,
    nationality STRING,
    url STRING
"""

In [0]:
constructors_df = (
    spark.read
        .format('json')
        .schema(constructors_schema)
        .option("mode", "FAILFAST")
        .load(source_file)
)

display(constructors_df)

In [0]:
constructors_final_df = add_ingestion_metadata(constructors_df)

display(constructors_final_df)

### Write to bronze delta table

In [0]:
write_to_bronze(
    input_df = constructors_final_df
    , target_table = table_name
    , batch_id = v_batch_id
)

In [0]:
%sql
select * from formula1_incr.bronze.constructors